<a href="https://colab.research.google.com/github/hnkwon02/baseball-analytics/blob/main/01_ssg_2026_fip/ssg_2026_fip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2026 SSG 투수진, 실력이었나 운이었나
### ERA와 FIP의 차이로 본 투수 평가

**분석 질문**
1. 2026 SSG 투수 중 ERA에 비해 실제 투구 내용(FIP)이 더 좋았던 투수는 누구인가? → 내년 반등 후보
2. 반대로 ERA가 FIP보다 지나치게 좋았던 투수는 누구인가? → 하락 위험
3. 삼진·볼넷 비율(K%, BB%)로 보면 SSG 투수진의 강점과 약점은 무엇인가?

**데이터**: FanGraphs KBO 리더보드 2026 시즌, 리그 전체 투수 (CSV로 저장)

## 0단계. 라이브러리 불러오기
Colab에는 pandas와 matplotlib이 이미 설치돼 있어서 바로 실행하면 됩니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## 1단계. 데이터 불러오기
Colab 왼쪽 폴더 아이콘으로 `kbo_2026_pitchers.csv`를 업로드한 뒤 실행하세요.

**중요**: SSG 투수만이 아니라 **리그 전체 투수**가 들어 있어야 합니다. FIP 계산에 리그 평균이 필요하기 때문이에요.

In [ ]:
df = pd.read_csv('kbo_2026_pitchers.csv')
print(df.shape)              # (투수 수, 컬럼 수)
print(df.columns.tolist())   # 컬럼 이름 확인
print(df['Team'].unique())   # 팀 표기 확인 — SSG가 어떻게 적혀 있는지 보기
df.head()

## 2단계. 이닝(IP) 표기 바꾸기 — 초보자가 가장 많이 틀리는 부분
야구 기록에서 `150.1`은 150.1이닝이 아니라 **150과 1/3이닝**이고, `150.2`는 150과 2/3이닝입니다.
그대로 계산하면 숫자가 틀어지므로 아웃카운트 기준의 실제 이닝으로 바꿔줍니다.

In [ ]:
def to_real_ip(ip):
    whole = int(ip)
    outs = round((ip - whole) * 10)   # .1 → 1아웃, .2 → 2아웃
    return whole + outs / 3

df['IP_real'] = df['IP'].apply(to_real_ip)
df[['Name', 'IP', 'IP_real']].head()

## 3단계. 리그 FIP 상수 구하기
FIP 공식:

$$FIP = \frac{13 \times HR + 3 \times (BB + HBP) - 2 \times SO}{IP} + C$$

상수 C는 **리그 전체 FIP 평균이 리그 ERA와 같아지도록** 맞추는 값이에요. 그래서 리그 전체 합계가 필요합니다.
리그 ERA는 (전체 자책점 × 9) ÷ 전체 이닝으로 구합니다. 자책점(ER) 컬럼이 이 계산에 필요해요.

In [ ]:
lg = df[['HR', 'BB', 'HBP', 'SO', 'ER', 'IP_real']].sum()

lg_era = lg['ER'] * 9 / lg['IP_real']
lg_core = (13 * lg['HR'] + 3 * (lg['BB'] + lg['HBP']) - 2 * lg['SO']) / lg['IP_real']
FIP_CONSTANT = lg_era - lg_core

print(f'2026 리그 ERA: {lg_era:.2f}')
print(f'2026 FIP 상수: {FIP_CONSTANT:.2f}')

## 4단계. 투수별 FIP, K%, BB% 계산
- **K%** = 삼진 ÷ 상대 타자 수(TBF)
- **BB%** = 볼넷 ÷ 상대 타자 수
- **ERA − FIP**: 양수가 크면 운이 나빴던 쪽(반등 후보), 음수가 크면 운이 좋았던 쪽(하락 위험)

In [ ]:
df['FIP'] = ((13 * df['HR'] + 3 * (df['BB'] + df['HBP']) - 2 * df['SO'])
             / df['IP_real'] + FIP_CONSTANT)
df['K%'] = df['SO'] / df['TBF'] * 100
df['BB%'] = df['BB'] / df['TBF'] * 100
df['K-BB%'] = df['K%'] - df['BB%']
df['ERA-FIP'] = df['ERA'] - df['FIP']

## 5단계. SSG 투수만 골라내기
이닝이 너무 적은 투수는 숫자가 크게 흔들리므로 최소 이닝 기준을 둡니다. 기준은 직접 정하고, 왜 그렇게 정했는지 해석에 한 줄 적어두세요.

In [ ]:
TEAM = 'SSG'      # 1단계에서 확인한 표기로 맞추기
MIN_IP = 30       # 최소 이닝 기준

ssg = df[(df['Team'] == TEAM) & (df['IP_real'] >= MIN_IP)].copy()
cols = ['Name', 'IP', 'ERA', 'FIP', 'ERA-FIP', 'K%', 'BB%', 'K-BB%']
ssg[cols].sort_values('ERA-FIP', ascending=False).round(2)

## 6단계. 시각화 — ERA vs FIP
대각선 위쪽(ERA가 FIP보다 높음)은 운이 나빴던 투수, 아래쪽은 운이 좋았던 투수입니다.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(ssg['FIP'], ssg['ERA'])
for _, r in ssg.iterrows():
    ax.annotate(r['Name'], (r['FIP'], r['ERA']), fontsize=8,
                xytext=(4, 4), textcoords='offset points')

lo = min(ssg['FIP'].min(), ssg['ERA'].min()) - 0.5
hi = max(ssg['FIP'].max(), ssg['ERA'].max()) + 0.5
ax.plot([lo, hi], [lo, hi], color='gray', lw=1, ls='--')
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_xlabel('FIP')
ax.set_ylabel('ERA')
ax.set_title('SSG pitchers 2026: ERA vs FIP')
plt.show()

## 7단계. 리그 평균과 비교
SSG 투수진 전체의 K%, BB%가 리그 평균보다 좋은지 나쁜지 봅니다.

In [ ]:
def staff_rates(d):
    return pd.Series({'K%': d['SO'].sum() / d['TBF'].sum() * 100,
                      'BB%': d['BB'].sum() / d['TBF'].sum() * 100})

compare = pd.DataFrame({'SSG': staff_rates(df[df['Team'] == TEAM]),
                        'KBO 평균': staff_rates(df)}).round(1)
compare

## 8단계. 해석 — 포트폴리오의 핵심
코치에게 보고하듯 **결론 → 근거 숫자 → 의미** 순서로 직접 쓰세요.

1. **반등 후보 (ERA−FIP 상위)**: 누구이고, 근거 숫자는?
2. **하락 위험 (ERA−FIP 하위)**: 누구이고, 왜 그렇게 판단했나?
3. **투수진 전체 성향**: 리그 평균 대비 K%, BB%로 본 강점·약점
4. **구단에 대한 제안**: 이 결과로 2027 시즌 준비에서 무엇을 우선해야 하나?

**한계**: FIP는 수비·구장 효과를 단순하게 처리하고, 이닝이 적은 투수는 신뢰도가 낮으며, 시즌 1년치라는 점을 적어두세요.